# Introduction

This notebook is dedicated to developing a robust, quantitative indicator for measuring and analyzing the realized volatility of a financial asset, specifically focusing on intraday movements. Unlike methods that rely solely on price direction, this analysis employs two cornerstone techniques: the Garman-Klass (GK) Estimator to accurately capture the true daily price dispersion (turmoil), and a statistically rigorous Z-Score normalization method to identify current volatility regimes. The primary goal is to transform raw price movement into an actionable signal that clearly distinguishes between periods of high market stress and unusual complacency, relative to a long-term historical average.

# 1. Data Loading

imports the required libraries (pandas, numpy, matplotlib) and loads the price data (Date, Open, High, Low, Close) from sp500.csv.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# ---------- Configuration ----------
TICKER = "AAPL"              # Change the ticker for the asset you want to analyze (should be available on Yahoo Finance).
START_DATE = "2010-01-01"
END_DATE = None               # None = up to today

DATE_COL = "Date"
OPEN_COL = "Open"
HIGH_COL = "High"
LOW_COL = "Low"
CLOSE_COL = "Close"

ROLL_WIN = 21
LONG_ROLL_WIN = 252
LAMBDA = 0.94
TRADING_DAYS = 252

OUT_CSV = f"{TICKER}_idv_gk_indicator.csv"
OUT_PNG = f"{TICKER}_idv_gk_indicator.png"

print("Fetching data from Yahoo Finance...")

# ---------- 1. Load and Validate Data ----------
df = yf.download(TICKER, start=START_DATE, end=END_DATE, group_by='ticker')

# Move index (Date) into a normal column
df.reset_index(inplace=True)

# --- Flatten MultiIndex columns if present ---
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [' '.join([str(c) for c in col if c]).strip() for col in df.columns.values]
else:
    df.columns = [str(c).strip() for c in df.columns]

# --- Clean up column names ---
# Remove the ticker name (like "^GSPC") and redundant spaces
df.columns = [col.replace(TICKER, '').replace('  ', ' ').strip() for col in df.columns]

# Rename 'Date' column if needed
for col in df.columns:
    if 'Date' in col:
        df.rename(columns={col: 'Date'}, inplace=True)

# Ensure required columns exist
required_cols = {DATE_COL, OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL}
assert required_cols.issubset(df.columns), \
    f"Data must contain all of {required_cols}. Found: {df.columns.tolist()}"

# Convert to numeric just in case
for col in [OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=required_cols)

# Ensure no zero/negative prices
df = df[(df[HIGH_COL] > 0) & (df[LOW_COL] > 0) & (df[OPEN_COL] > 0) & (df[CLOSE_COL] > 0)]

print("Data loaded and validated successfully")
print(df.head())


Fetching data from Yahoo Finance...


C:\Users\ahmed\AppData\Local\Temp\ipykernel_13692\3181432614.py:28: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

Data loaded and validated successfully
        Date         Open         High          Low        Close      Volume
0 2010-01-04  1116.560059  1133.869995  1116.560059  1132.989990  3991400000
1 2010-01-05  1132.660034  1136.630005  1129.660034  1136.520020  2491020000
2 2010-01-06  1135.709961  1139.189941  1133.949951  1137.140015  4972660000
3 2010-01-07  1136.270020  1142.459961  1131.319946  1141.689941  5270680000
4 2010-01-08  1140.520020  1145.390015  1136.219971  1144.979980  4389590000


# 2. Garman-Klass Volatility Estimation

Calculates the daily volatility (standard deviation) using the Garman-Klass (GK) estimator. This formula uses the log of the daily range (High/Low) and the log of the intraday return (Close/Open).

$$\sigma_{GK}^2 = 0.5 \times [\ln(H/L)]^2 - [2\ln(2) - 1] \times [\ln(C/O)]^2$$

The GK method provides a more accurate estimate of true daily market volatility (price dispersion) than simpler measures like close-to-close returns because it captures intraday movements.

In [14]:
# ---------- 2. Calculate Robust Daily Volatility (Garman-Klass) ----------

# Calculate log-return components
ln_hl = np.log(df[HIGH_COL] / df[LOW_COL])
ln_co = np.log(df[CLOSE_COL] / df[OPEN_COL])

# Calculate Garman-Klass daily variance
# Formula: 0.5 * [ln(H/L)]^2 - [2*ln(2) - 1] * [ln(C/O)]^2
gk_var = 0.5 * (ln_hl**2) - (2 * np.log(2) - 1) * (ln_co**2)

# Get daily volatility (std dev) and clip at 0 to avoid sqrt(negative)
df["daily_vol_gk"] = np.sqrt(gk_var.clip(lower=0))



# 3. Smoothing and Annualization

The code takes the daily GK volatility and smooths it using a 21-day Rolling Mean and a RiskMetrics EWMA (Exponentially Weighted Moving Average). It then annualizes the result ($\times \sqrt{252}$).

Smoothing reveals the underlying trend. Annualization is the standard financial practice for comparing volatility figures across different time periods and assets.

In [15]:
# ---------- 3. Compute Smoothed Volatility (Rolling & EWMA) ----------
# We take the .mean() of our daily volatility estimates

# Rolling (simple) mean of daily vol
df["sigma_roll_daily"] = df["daily_vol_gk"].rolling(ROLL_WIN, min_periods=ROLL_WIN//2).mean()

# EWMA (exponential) mean of daily vol
df["sigma_ewma_daily"] = df["daily_vol_gk"].ewm(alpha=(1 - LAMBDA), min_periods=ROLL_WIN//2, adjust=False).mean()

# Annualize and convert to percentage for plotting
df["sigma_roll_ann"] = df["sigma_roll_daily"] * np.sqrt(TRADING_DAYS) * 100
df["sigma_ewma_ann"] = df["sigma_ewma_daily"] * np.sqrt(TRADING_DAYS) * 100

print("Building actionable oscillator...")



Building actionable oscillator...


# 4. Z-Score Regime Identification

Calculates the Z-Score of the smoothed EWMA volatility against its 252-day (one-year) moving average and standard deviation.

This step normalizes the volatility. Volatility is relative; a 20% annualized vol might be high one year and normal another. The Z-Score tells how many standard deviations the current vol is away from its historical norm, effectively identifying statistically extreme "panic" (high Z-Score) or "complacency" (low Z-Score) market regimes.

In [16]:

# ---------- 4. Create Meaningful Oscillator (Z-Score) ----------
# Values > 2.0 can be considered "panic"
# Values < -1.0 can be considered "complacency"

# Long-term mean and std dev of the daily EWMA vol
long_mean = df["sigma_ewma_daily"].rolling(LONG_ROLL_WIN, min_periods=LONG_ROLL_WIN//2).mean()
long_std = df["sigma_ewma_daily"].rolling(LONG_ROLL_WIN, min_periods=LONG_ROLL_WIN//2).std()

# IDV Z-Score
df["IDV_ZScore"] = (df["sigma_ewma_daily"] - long_mean) / long_std



# 5. Plotting and Output

Generates a dual-axis chart, saving the final calculated data to a CSV (sp500_gk_analysis.csv) and the plot to a PNG (sp500_gk_analysis.png).

This makes the complex analytical results easy to visualize and interpret for financial decision-making, while the CSV allows the data to be used by other applications.

In [17]:

# ---------- 5. Save Results ----------
df_out = df[[
    DATE_COL, "daily_vol_gk", "sigma_roll_ann", "sigma_ewma_ann", "IDV_ZScore"
]].copy()
df_out.to_csv(OUT_CSV, index=False, float_format="%.4f")

print("Generating new chart...")
# ---------- 6. Plot Improved Indicator ----------
plt.style.use("seaborn-v0_8")
fig, ax1 = plt.subplots(figsize=(12, 5))
fig.suptitle(f"{TICKER} Intra-Day Volatility (Garman-Klass Estimator)")

# --- Left Axis: Annualized Volatility (%) ---
ax1.plot(df[DATE_COL], df["sigma_ewma_ann"], color="#1f77b4", label=f"EWMA Vol (ann.) (λ={LAMBDA})")
ax1.plot(df[DATE_COL], df["sigma_roll_ann"], color="#ff7f0e", alpha=0.7, label=f"Rolling Vol {ROLL_WIN} (ann.)")
ax1.set_ylabel("Annualized Volatility (%)")
ax1.set_xlabel("Date")
ax1.legend(loc="upper left")
ax1.set_ylim(bottom=0) # Volatility cannot be negative

# --- Right Axis: Actionable Z-Score Oscillator ---
ax2 = ax1.twinx()
ax2.plot(df[DATE_COL], df["IDV_ZScore"], color="#2ca02c", alpha=0.6, label="IDV Z-Score (vs 1-Yr)")

# Add meaningful statistical thresholds
ax2.axhline(2.0, color="red", ls="--", lw=1, label="High Volatility (Z > 2.0)")
ax2.axhline(0.0, color="gray", ls="--", lw=1)
ax2.axhline(-1.0, color="blue", ls="--", lw=1, label="Low Volatility (Z < -1.0)")

ax2.set_ylabel(f"IDV Z-Score (vs. {LONG_ROLL_WIN}-day mean)")
ax2.legend(loc="upper right")

# --- Finalize ---
fig.tight_layout()
plt.savefig(OUT_PNG, dpi=150)
plt.close()

print(f"Successfully saved: {OUT_CSV} and {OUT_PNG}")

Generating new chart...
Successfully saved: ^GSPC_idv_gk_indicator.csv and ^GSPC_idv_gk_indicator.png


# 5b. Bonus Interactive Chart

In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Generating interactive chart (fixed threshold placement)...")

fig = make_subplots(
    specs=[[{"secondary_y": True}]]
)

# Left axis traces (annualized vol)
fig.add_trace(
    go.Scatter(x=df[DATE_COL], y=df["sigma_ewma_ann"],
               mode="lines", name=f"EWMA Vol (ann.) (λ={LAMBDA})"),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=df[DATE_COL], y=df["sigma_roll_ann"],
               mode="lines", name=f"Rolling Vol {ROLL_WIN} (ann.)",
               line=dict(dash="dot")),
    secondary_y=False
)

# Right axis trace (IDV Z-Score)
fig.add_trace(
    go.Scatter(x=df[DATE_COL], y=df["IDV_ZScore"],
               mode="lines", name="IDV Z-Score (vs 1-Yr)"),
    secondary_y=True
)

# --- Threshold lines ON SECONDARY Y-AXIS ---
x0 = df[DATE_COL].min()
x1 = df[DATE_COL].max()

thresholds = [
    {"y": 2.0,  "color": "red",  "text": "High Vol (Z > 2.0)"},
    {"y": 0.0,  "color": "gray", "text": "Zero"},
    {"y": -1.0, "color": "blue", "text": "Low Vol (Z < -1.0)"}
]

for t in thresholds:
    fig.add_shape(
        type="line",
        x0=x0, x1=x1,
        y0=t["y"], y1=t["y"],
        xref="x", yref="y2",          # <-- important: place on secondary y-axis
        line=dict(color=t["color"], width=1, dash="dash")
    )
    # add annotation near the right side of the chart, anchored to y2
    fig.add_annotation(
        x=x1,
        y=t["y"],
        xref="x",
        yref="y2",
        text=t["text"],
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        bgcolor="rgba(28, 28, 28, 1)",
        font=dict(size=11)
    )

# Layout
fig.update_layout(
    title=f"{TICKER} Intra-Day Volatility (Garman–Klass Estimator)",
    template="plotly_dark",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=60, r=60, t=80, b=50),
    width=1000, height=550
)

fig.update_yaxes(title_text="Annualized Volatility (%)", secondary_y=False, rangemode="tozero")
fig.update_yaxes(title_text=f"IDV Z-Score (vs. {LONG_ROLL_WIN}-day mean)", secondary_y=True)

# Save and show
OUT_HTML = OUT_PNG.replace(".png", ".html")
fig.write_html(OUT_HTML)
print(f"✅ Saved interactive chart to {OUT_HTML}")
fig.show()


Generating interactive chart (fixed threshold placement)...
✅ Saved interactive chart to ^GSPC_idv_gk_indicator.html


# Conclusion

The application of the Garman-Klass estimator, coupled with the Z-Score oscillator, successfully yields a powerful tool for quantitative regime identification. The resulting Z-Score line serves as a statistically normalized metric, allowing analysts to objectively confirm whether the market is currently experiencing volatility that is a significant deviation (e.g., two standard deviations) from its 252-day norm. The indicator's primary utility lies not in prediction, but in regime definition, providing context that is essential for position sizing, risk management, and strategy adaptation.

### Further Research Opportunities:

- Overnight Gap Analysis: Incorporate an explicit measure of the close-to-open gap to capture total volatility, which the intraday GK estimator inherently excludes.

- Asymmetric Volatility: Investigate whether high-volatility spikes ($Z > 2.0$) or low-volatility compressions ($Z < -1.0$) offer reliable mean-reversion or trend-following entry signals.

- Parameter Optimization: Test the stability and signal quality by adjusting the $\lambda$ parameter for the EWMA or varying the $252$-day lookback window for the Z-Score calculation.